# Customer Segmentation — RFM and CLV


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

CLEAN = Path("../data/cleaned")
tx = pd.read_csv(CLEAN/"transactions_clean.csv", parse_dates=["transaction_date"])
tx.head()


## RFM calculation


In [ ]:
snapshot = tx["transaction_date"].max() + pd.Timedelta(days=1)

rfm = tx.groupby("customer_id").agg(
    last_purchase_date=("transaction_date","max"),
    frequency=("transaction_id","nunique"),
    monetary=("total_amount","sum")
).reset_index()

rfm["recency_days"] = (snapshot - rfm["last_purchase_date"]).dt.days

rfm["R_score"] = pd.qcut(rfm["recency_days"].rank(method="first"), 5, labels=[5,4,3,2,1]).astype(int)
rfm["F_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
rfm["M_score"] = pd.qcut(rfm["monetary"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)

rfm["RFM_score"] = rfm["R_score"].astype(str)+rfm["F_score"].astype(str)+rfm["M_score"].astype(str)
rfm.head()


## Segment logic


In [ ]:
def segment(row):
    r, f, m = row.R_score, row.F_score, row.M_score
    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    if r >= 4 and f >= 3:
        return "Loyal Customers"
    if r >= 4 and m >= 4:
        return "High Potential"
    if r <= 2 and f >= 3:
        return "At Risk"
    if r <= 2 and f <= 2:
        return "Hibernating"
    if r >= 3 and f <= 2:
        return "New/Promising"
    return "Regular"

rfm["segment"] = rfm.apply(segment, axis=1)
rfm["segment"].value_counts()


## CLV proxy


In [ ]:
# Portfolio project approximation rather than a production survival model.
margin_rate = 0.25
retention_years = 2

rfm["annual_revenue_proxy"] = rfm["monetary"]
rfm["estimated_clv"] = rfm["annual_revenue_proxy"] * margin_rate * retention_years

rfm.sort_values("estimated_clv", ascending=False).head(20)


## Segment economics


In [ ]:
segment_summary = rfm.groupby("segment").agg(
    customers=("customer_id","nunique"),
    revenue=("monetary","sum"),
    avg_revenue=("monetary","mean"),
    avg_recency=("recency_days","mean"),
    avg_frequency=("frequency","mean"),
    estimated_clv=("estimated_clv","sum")
).sort_values("revenue", ascending=False)

segment_summary


## Export


In [ ]:
rfm.to_csv(CLEAN/"customer_rfm.csv", index=False)
print("Saved:", CLEAN/"customer_rfm.csv")
